# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-34: Rice iPPG (HRVCam) Phase-3 Preprocessing

### What this notebook does
Writes the Rice iPPG dataset into one Phase-3 store in the unified schema (`p3_v1`), directly
in the rechunked uncompressed layout, as an **evaluation-only** corpus for heart rate and
heart-rate variability. Every frame is read once and used three times: the face crop the
training contract asks for, the crop the app's own box would take today, and the mean colour
of three skin regions. The reference comes from NB_P3_33 - the oximeter's rebuilt clock, the
beats from the steepest upstroke, and each recording's own offset to the camera - and is
stored both on the frame grid and natively.

No estimator is scored here. The POS readings in the store are the app's own, kept as quality
numbers, and the store reproduces NB_P3_33's figures before anything downstream runs.

### Decisions carried from NB_P3_33 (all measured there)
- **All 70 recordings** carry heart rate; **58 carry HRV**, the other twelve set aside because
  the positive control doubted their oximeter clock or they keep too few intervals. The lists
  are carried here and checked against a recomputation.
- **Frames are dated by the camera's clock** at each recording's own measured rate
  (30.017-30.018 fps), mapped onto the PC's clock through the lower envelope of the arrival
  stamps. A uniform grid at that rate is never more than 0.046 ms off.
- **The mosaic is RGGB**, read from the data, which is OpenCV's `COLOR_BayerBG2RGB`.
- **The fiducial is the steepest point of the upstroke, refined between samples**, on both
  sides. The systolic peak moves between the systolic and diastolic waves in seven recordings.
- **Each recording's camera-to-finger offset** is searched within 0.25 s of its subject's
  anchor, the anchors themselves carried from section 13: a pooled search needs all five of a
  subject's recordings at once, and this batch writes one recording at a time.
- **The app's framing gate refuses all 70 recordings**, because a face filling 60-95% of the
  frame's height cannot fit a box padded by 0.6. That is why the store keeps two crops.

### The two crops, and why both
There is no single faithful crop for this dataset, so the choice is left to whoever evaluates
it rather than settled by this notebook:
- `frames` pads the frame with a replicated border until the whole padded box fits. The face
  keeps the share of the crop and the aspect it has in training; what enters at the edges is a
  static border with no pulse of its own.
- `frames_clamped` is the box clamped to the frame, which is what the app does today. The face
  then fills more of the crop than the contract assumes, and the crop's aspect leaves the
  training range.

Both are written for every recording, so the comparison costs a read of the store rather than
a second pass over 254,352 raw frames.

### Colour
The crops hold raw demosaiced values, with the skin's own balance recorded per recording
(`skin_r_g`, `skin_b_g`). A fixed per-channel gain is not a decision this store has to take:
PhysNet z-scores each channel over the clip and POS divides each channel by its own mean, so
white balance cancels in both. Gamma does not cancel, but it is a lookup on uint8 and can be
applied when the frames are read.

### Two guards against this store being trained on
The roles decided in NB_P3_33 are evaluation ones. Unlike ECG-Fitness this corpus does have a
pulse reference, so nothing about it is inherently untrainable - which is exactly why the
decision is made explicit here:
1. **It is written to `phase 3 benchmarks\`, not `phase 3 datasets\`**, outside every training
   glob.
2. **No `cardiac_sqi` attribute.** The training loader skips a group whose `cardiac_sqi` falls
   below 0.05 and reads a missing one as 0.0. The reference's own quality lives in
   `ref_kept_share`, `ref_rmssd_ms` and `has_hrv` instead.

Section 4 applies the loader's own filter to the store and requires that it admit nothing.
Adding this corpus to training later is a decision to take in the open, not by moving a file.

## Imports & Setup

In [6]:
import json
import os
import re
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import h5py
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.optimize import linprog
from scipy.signal import butter, find_peaks, sosfiltfilt, welch

# Walk up from the notebook until the repo root is found, identified by the MediaPipe asset (the
# same test NB_P3_33 uses; cwd is the nested notebook folder).
MP_REL = Path('app/live_hr_demo/models/mediapipe/face_landmarker.task')
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / MP_REL).exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'no {MP_REL} in any parent of {Path.cwd()}')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# The app's code is imported, not restated: the box, the crop, the gates and the reading below are
# the ones the app would produce from these frames, and they move if the app moves.
from app.live_vitals.capture.motion import motion_verdict
from app.live_vitals.capture.session import (detection_positions, face_verdict, framing_verdict,
                                             stable_box)
from app.live_vitals.config import (CLIP_LEN, CROP_PADDING, MAX_BOX_SIDE_LOST, MIN_CONFIDENCE,
                                    N_DETECT_FRAMES, SPECTRAL_ROI_FRACTION, TARGET_FPS, TARGET_SIZE,
                                    WINDOW_STRIDE)
from app.live_vitals.estimators.hr_spectral import HRSpectral
from app.live_vitals.preprocess.face_box import make_landmarker
from app.live_vitals.preprocess.frames import crop_resize
from app.live_vitals.signal.spectral import pos, rgb_trace

RICE_ROOT = Path(r'F:\Rice iPPG Dataset')
DATASET_NAME = 'rice_ippg'
# A benchmark folder, not `phase 3 datasets`: the first of the two guards below.
OUT_DIR = Path(r'D:\QualityPhys\phase 3 benchmarks')
OUT_H5 = OUT_DIR / f'{DATASET_NAME}_phase3_rc_none.h5'   # rechunked layout, written directly
LOG_PATH = REPO_ROOT / 'Data' / f'processing_log_{DATASET_NAME}_phase3.csv'   # recording-level only

ACTIVITIES = ('stationary', 'reading', 'watching', 'talking', 'deep')   # protocol order
FRAME_DIR = 'cam_flea3_0'
FRAME_W, FRAME_H = 1280, 1024
PGM_HEADER = b'P5\n1280 1024\n255\n'   # checked on every frame read, never assumed
PGM_BYTES = FRAME_W * FRAME_H + len(PGM_HEADER)
CYCLE_WRAP_S = 128.0        # the camera clock's seconds field wraps every 128 s
PATTERN = 'RGGB'            # the mosaic, read from the data in NB_P3_33 section 9
BAYER_CODE = cv2.COLOR_BayerBG2RGB   # OpenCV's code for RGGB: it names the cell from another pixel
SITES = {'R': [(0, 0)], 'G': [(0, 1), (1, 0)], 'B': [(1, 1)]}   # (row, column) inside a 2x2 cell

OX_FS = 60.0                # CMS50D+ waveform rate, measured in NB_P3_33 section 3
OX_TOP_BIT = 128            # every waveform byte arrives with its top bit set
OX_FULL_SCALE = 127         # top of the 7-bit waveform; 0 and 127 are clipped values
LOSS_LOOKAHEAD = 30         # samples (0.5 s) that must all arrive a period late to accept a loss
LOSS_TOL_S = 0.003          # slack on that test, for jitter in the floor itself
HR_MIN_BPM, HR_MAX_BPM = 35.0, 180.0   # physiological limits on a beat interval
BEAT_SPACING = 0.55         # minimum spacing between beats, as a share of the dominant period
IBI_LOCAL_TOL = 0.25        # an interval this far from the median around it is an artefact
GAP_GUARD = 2               # samples either side of a beat that must be real, not interpolated

# Skin regions as fractions of the landmark box (x0, y0, x1, y1), left and right as in the image,
# exactly as NB_P3_33 part 3 placed them: the forehead between the top of the landmarks and the
# brows, each cheek between the eye and the nostril.
REGIONS = {'forehead': (0.30, 0.05, 0.70, 0.20),
           'cheek_left': (0.14, 0.50, 0.36, 0.68),
           'cheek_right': (0.64, 0.50, 0.86, 0.68)}
MIN_REGION_CELLS = 1000     # a region needs this many 2x2 cells inside the frame to be read

MATCH_TOL_S = 0.15          # a camera beat matches a reference beat this close
ALIGN_HALF_WIDTH_S = 0.25   # each recording is aligned within this of its subject's anchor
# Subject anchors from NB_P3_33 section 13, each found by searching that subject's five recordings
# together over +/-2 s. Carried rather than recomputed: a pooled search needs every recording of a
# subject at once, and the batch below writes one recording at a time.
ANCHOR_MS = {'003': -7.5, '004': 7.5, '005': 22.5, '006': -35.0, '007': -37.5, '008': -60.0,
             '009': -67.5, '010': -37.5, '011': -27.5, '012': -22.5, '013': -2.5, '014': -37.5,
             '015': -30.0, '016': -42.5}
# The references NB_P3_33 section 5 set aside for HRV, and why. Carried, and checked in section 4.
CLOCK_DOUBTED = {('005', 'stationary'), ('005', 'reading'), ('005', 'watching'),
                 ('006', 'stationary'), ('006', 'reading'), ('013', 'watching'),
                 ('015', 'watching')}
TOO_THIN = {('004', 'deep'), ('004', 'watching'), ('005', 'talking'), ('006', 'stationary'),
            ('008', 'watching'), ('013', 'watching'), ('014', 'watching')}
HRV_EXPECTED = 58           # recordings left for HRV once both sets are removed

FRAME_DARK, FRAME_BRIGHT = 5.0, 250.0   # the exposure gate, as every Phase-3 store
SCHEMA_VERSION = 'p3_v1'
FRAMES_PER_CHUNK = 32       # frames per HDF5 chunk, for both frame datasets
FRAME_WORKERS = 8           # frames read, demosaiced and cropped in parallel threads
OVERWRITE = False
MAX_RECORDS = None          # set to an int for a trial run

LANDMARKER = make_landmarker()
print('Rice iPPG root:', RICE_ROOT, '| exists:', RICE_ROOT.exists())
print('output store:', OUT_H5, '(exists)' if OUT_H5.exists() else '(new)')
print(f'app contract: crop {TARGET_SIZE}x{TARGET_SIZE} from a box padded by {CROP_PADDING:g}, colour '
      f'trace from its central {SPECTRAL_ROI_FRACTION:.0%}, windows of {CLIP_LEN} frames every '
      f'{WINDOW_STRIDE}, confidence gate {MIN_CONFIDENCE:g}, framing gate {MAX_BOX_SIDE_LOST:.0%} of '
      f'the box side, training rate {TARGET_FPS:g} fps')

Rice iPPG root: F:\Rice iPPG Dataset | exists: True
output store: D:\QualityPhys\phase 3 benchmarks\rice_ippg_phase3_rc_none.h5 (exists)
app contract: crop 72x72 from a box padded by 0.6, colour trace from its central 35%, windows of 160 frames every 80, confidence gate 0.65, framing gate 2% of the box side, training rate 30 fps


## 1. The reference, as NB_P3_33 built it
The same code as NB_P3_33 part 1, restated so this notebook stands alone: the frame clock from
the camera's own counter, the oximeter's clock rebuilt from its read stamps with the lost
samples placed, the waveform on the device's 60 Hz grid, and the beats and interval rules that
decide what HRV is computed from. Nothing here reads a frame; the pass over all 70 recordings
takes about five minutes, most of it placing samples in their slots.

In [7]:
SUBJECT_DIR = re.compile(r'^Subj_(\d{3})$')
RECORDING_DIR = re.compile(r'^Subj_(\d{3})_([a-z]+)$')
BEAT_BAND = butter(2, [0.5, 8.0], btype='band', fs=OX_FS, output='sos')
HR_BAND = butter(2, [HR_MIN_BPM / 60.0, HR_MAX_BPM / 60.0], btype='band', fs=OX_FS, output='sos')


def frame_path(folder, index):
    return folder / FRAME_DIR / f'Frame{index:05d}.pgm'


def read_raw(path):
    """One raw Bayer frame as a 2-D uint8 array, with its header checked."""
    data = path.read_bytes()
    if not data.startswith(PGM_HEADER) or len(data) != PGM_BYTES:
        raise ValueError(f'{path.name} is not one 8-bit {FRAME_W}x{FRAME_H} frame')
    return np.frombuffer(data, dtype=np.uint8, offset=len(PGM_HEADER)).reshape(FRAME_H, FRAME_W)


def scan_recordings(root=RICE_ROOT):
    """One row per recording folder, with its frame count. Every subject folder holds a second
    folder of the same name, and the recordings sit inside that one."""
    rows = []
    for subj in sorted(p for p in root.iterdir() if p.is_dir() and SUBJECT_DIR.match(p.name)):
        inner = subj / subj.name
        if not inner.is_dir():
            continue
        for folder in sorted(p for p in inner.iterdir()
                             if p.is_dir() and RECORDING_DIR.match(p.name)):
            subject, activity = RECORDING_DIR.match(folder.name).groups()
            sizes = [e.stat().st_size for e in os.scandir(folder / FRAME_DIR)
                     if e.name.endswith('.pgm')]
            rows.append({'subject': subject, 'activity': activity, 'folder': folder,
                         'frames': len(sizes), 'odd_size': sum(s != PGM_BYTES for s in sizes)})
    return pd.DataFrame(rows)


def lower_line(x, y):
    """(a, b) of the line a + b*x that lies under every point with the least total gap.

    Times written by a PC are late by a variable amount and never early, so a clock relation is
    the lower envelope of the points. With x centred, the total gap depends only on the line's
    height at the centre, so the programme maximises that height.
    """
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    x0, y0 = x.mean(), y.min()
    xc = x - x0
    sol = linprog(c=[-1.0, 0.0], A_ub=np.column_stack([np.ones_like(xc), xc]), b_ub=y - y0,
                  bounds=[(None, None), (None, None)], method='highs')
    if not sol.success:
        raise RuntimeError(f'lower envelope fit failed: {sol.message}')
    a, b = sol.x
    return float(y0 + a - b * x0), float(b)


def read_camera_logs(folder):
    """Returns (PC arrival times, camera clock from the first frame, frame counter), in s."""
    host = np.loadtxt(folder / 'CameraTimeLog0.txt', ndmin=1)
    cycles = np.loadtxt(folder / 'CameraCyclesLog0.txt', ndmin=1)
    with open(folder / 'CameraFrame0.txt', encoding='utf-8', errors='replace') as f:
        counter = np.array([int(re.search(r'\d+', line).group()) for line in f if line.strip()])
    step = np.diff(cycles)
    step[step < 0] += CYCLE_WRAP_S
    return host, np.concatenate([[0.0], np.cumsum(step)]), counter


def frame_clock(folder):
    """Frame times on the PC clock, the frame rate on the camera's clock, and the camera's drops."""
    host, clock, counter = read_camera_logs(folder)
    a, b = lower_line(clock, host - clock)
    counter_step = np.diff(counter)
    return (a + (1.0 + b) * clock, (len(clock) - 1) / float(clock[-1]),
            int((counter_step[counter_step > 0] - 1).sum()))


def read_oximeter(folder):
    """Returns (waveform bytes, PC read times in s) from px1_full.mat, the complete record."""
    mat = sio.loadmat(folder / 'PulseOX' / 'px1_full.mat', squeeze_me=True)
    return (np.atleast_1d(mat['pulseOxRecord']).astype(np.int64),
            np.atleast_1d(mat['pulseOxTime']).astype(np.float64))


def sample_slots(stamps, fs=OX_FS, lookahead=LOSS_LOOKAHEAD, tol=LOSS_TOL_S):
    """Places every read sample in its slot on the device's own grid.

    slots[k] is k plus the samples lost before sample k. A stamp minus its slot time sits on a
    floor of prompt reads, and the floor steps up one period at each lost sample. A loss is
    accepted only when all of the next `lookahead` samples sit at least a period (less `tol`)
    above the floor of the `lookahead` before them, so one late read never passes for a loss.
    """
    period = 1.0 / fs
    n = stamps.size
    slots = np.empty(n, dtype=np.int64)
    floor = np.empty(n)
    lost = 0
    for k in range(n):
        hi = min(n, k + lookahead)
        ahead = stamps[k:hi] - (np.arange(k, hi) + lost) * period
        before = ahead.min() if k == 0 else floor[max(0, k - lookahead):k].min()
        while hi - k >= 5 and ahead.min() - before >= period - tol:
            lost += 1
            ahead = ahead - period
        slots[k] = k + lost
        floor[k] = stamps[k] - slots[k] * period
    return slots


def rebuild_clock(stamps, fs=OX_FS):
    """Slots, sample times on the PC clock, read delays and the device's measured rate."""
    slots = sample_slots(stamps, fs)
    grid_s = slots / fs
    a, b = lower_line(grid_s, stamps - grid_s)
    times = a + (1.0 + b) * grid_s
    return {'slots': slots, 'times': times, 'delay': stamps - times, 'rate_hz': fs / (1.0 + b)}


def on_grid(wave, slots):
    """The waveform on the device's 60 Hz grid, with lost samples interpolated and marked."""
    grid = np.full(int(slots[-1]) + 1, np.nan)
    grid[slots] = wave
    missing = np.isnan(grid)
    if missing.any():
        idx = np.arange(grid.size)
        grid[missing] = np.interp(idx[missing], idx[~missing], grid[~missing])
    return grid, missing


def dominant_period(signal, fs):
    """The dominant beat period in samples, from a 16 s Welch spectrum."""
    f, p = welch(signal - signal.mean(), fs=fs, nperseg=min(len(signal), int(16 * fs)))
    band = (f >= HR_MIN_BPM / 60.0) & (f <= HR_MAX_BPM / 60.0)
    return fs / f[band][np.argmax(p[band])]


def refine(y, i):
    """Sub-sample position of a maximum from a parabola through three points. The index comes back
    unchanged at an edge, on a flat top, or when the vertex would leave the sample's cell."""
    if i <= 0 or i >= len(y) - 1:
        return float(i)
    left, centre, right = float(y[i - 1]), float(y[i]), float(y[i + 1])
    denominator = left - 2.0 * centre + right
    if abs(denominator) < 1e-12:
        return float(i)
    offset = 0.5 * (left - right) / denominator
    return float(i) + (offset if abs(offset) <= 0.5 else 0.0)


def steepest_rises(signal, fs, period):
    """Sub-sample positions of the steepest upstroves, spaced by the dominant period.

    A candidate must climb at least 40% as steeply as the median of the candidates around it,
    which drops ripples on the downstroke without a fixed threshold that the breathing-driven
    amplitude swing would defeat.
    """
    slope = np.gradient(signal)
    idx, _ = find_peaks(slope, distance=max(1, int(BEAT_SPACING * period)))
    if idx.size < 3:
        return np.array([]), slope
    local = pd.Series(slope[idx]).rolling(9, center=True, min_periods=3).median().to_numpy()
    idx = idx[slope[idx] >= 0.4 * local]
    return np.array([refine(slope, int(i)) for i in idx]), slope


def find_beats(grid, missing, fs=OX_FS):
    """One row per reference beat: the steepest-upstroke time, and whether it borders a lost or
    clipped sample."""
    x = sosfiltfilt(BEAT_BAND, grid - grid.mean())
    period = dominant_period(sosfiltfilt(HR_BAND, grid - grid.mean()), fs)
    positions, _ = steepest_rises(x, fs, period)
    rows = []
    for p in positions:
        i = int(round(p))
        around = slice(max(0, i - GAP_GUARD), i + GAP_GUARD + 1)
        rows.append({'slope_s': p / fs, 'near_gap': bool(missing[around].any()),
                     'clipped': bool(((grid[around] <= 0) | (grid[around] >= OX_FULL_SCALE)).any())})
    return pd.DataFrame(rows, columns=['slope_s', 'near_gap', 'clipped']), period


def intervals(beats):
    """Beat-to-beat intervals with the artefact rules applied. `kept` marks the intervals HRV is
    computed from, in the order the rules are applied."""
    t = beats.slope_s.to_numpy()
    ibi = np.diff(t)
    flagged = (beats.near_gap | beats.clipped).to_numpy()
    in_range = (ibi > 60.0 / HR_MAX_BPM) & (ibi < 60.0 / HR_MIN_BPM)
    clean = ~flagged[:-1] & ~flagged[1:]
    candidate = in_range & clean
    local = (pd.Series(np.where(candidate, ibi, np.nan))
             .rolling(7, center=True, min_periods=3).median().to_numpy())
    consistent = np.abs(ibi - local) <= IBI_LOCAL_TOL * local
    return pd.DataFrame({'t': t[1:], 'ibi': ibi, 'kept': candidate & consistent})


def hrv_metrics(table):
    """Mean heart rate, SDNN and RMSSD from the kept intervals."""
    ibi, kept = table.ibi.to_numpy(), table.kept.to_numpy()
    pair = kept[1:] & kept[:-1]
    diffs, values = np.diff(ibi)[pair], ibi[kept]
    if values.size < 8 or diffs.size < 8:
        return dict.fromkeys(('hr_bpm', 'sdnn_ms', 'rmssd_ms'), np.nan)
    return {'hr_bpm': 60.0 / values.mean(), 'sdnn_ms': values.std(ddof=1) * 1000.0,
            'rmssd_ms': float(np.sqrt(np.mean(diffs ** 2)) * 1000.0)}


def reference_of(row):
    """Everything the reference contributes for one recording: the frames' clock, the waveform on
    its rebuilt clock, the beats on the PC clock, and the recording-level reference numbers."""
    frame_t, fps, dropped = frame_clock(row.folder)
    values, stamps = read_oximeter(row.folder)
    clock = rebuild_clock(stamps)
    wave = (values - OX_TOP_BIT).astype(np.float64)
    grid, missing = on_grid(wave, clock['slots'])
    grid_times = np.interp(np.arange(grid.size), clock['slots'], clock['times'])
    beats, _ = find_beats(grid, missing)
    table = intervals(beats)
    beat_t = np.interp(beats.slope_s.to_numpy() * OX_FS, clock['slots'], clock['times'])
    covered = float(grid_times[-1] - frame_t[0])
    return {
        'frame_t': frame_t, 'fps': fps, 'dropped': dropped,
        'grid': grid, 'grid_t': grid_times, 'missing': missing,
        'beat_t': beat_t, 'beat_kept': table.kept.to_numpy(),
        'metrics': hrv_metrics(table),
        'row': {'subject': row.subject, 'activity': row.activity, 'frames': int(row.frames),
                'odd_size': int(row.odd_size), 'fps': fps, 'dropped': dropped,
                'ox_samples': int(stamps.size),
                'ox_lost': int((np.diff(clock['slots']) - 1).sum()),
                'ox_rate_hz': clock['rate_hz'],
                'ox_delay_med_ms': float(np.median(clock['delay']) * 1000.0),
                'ref_starts_after_s': float(grid_times[0] - frame_t[0]),
                'ref_covers_s': covered, 'ref_short_by_s': float(frame_t[-1] - grid_times[-1]),
                'ref_beats': int(len(beats)), 'ref_kept_share': float(table.kept.mean()),
                'ref_kept_s': float(table.loc[table.kept, 'ibi'].sum()),
                **{f'ref_{k}': v for k, v in hrv_metrics(table).items()}}}


recordings = scan_recordings()
REF = {}
t_ref = time.perf_counter()
for r in recordings.itertuples():
    REF[(r.subject, r.activity)] = reference_of(r)
reference = pd.DataFrame([REF[k]['row'] for k in REF])
reference['has_hrv'] = [(s, a) not in CLOCK_DOUBTED | TOO_THIN
                        for s, a in zip(reference.subject, reference.activity)]

print(f'{len(recordings)} recordings, {recordings.frames.sum()} raw frames, read in '
      f'{time.perf_counter() - t_ref:.0f} s | frames that are not one 8-bit frame by size: '
      f'{int(recordings.odd_size.sum())} | frames dropped by the camera counter: '
      f'{int(reference.dropped.sum())}')
print(f'frame rate on the camera clock {reference.fps.min():.4f}-{reference.fps.max():.4f} fps | '
      f'oximeter rate {reference.ox_rate_hz.min():.4f}-{reference.ox_rate_hz.max():.4f} Hz | '
      f'samples lost {int(reference.ox_lost.sum())}')
print(f'the reference starts {reference.ref_starts_after_s.min():.2f}-'
      f'{reference.ref_starts_after_s.max():.2f} s after the first frame; it ends before the last '
      f'frame by more than 1 s in '
      f'{int((reference.ref_short_by_s > 1).sum())} recordings '
      f'{reference.loc[reference.ref_short_by_s > 1, ["subject", "activity", "ref_short_by_s"]].round(1).to_dict("records")}')
print(f'references that carry HRV: {int(reference.has_hrv.sum())} of {len(reference)} '
      f'(NB_P3_33 section 5: {HRV_EXPECTED})')
print(reference[reference.has_hrv].groupby('activity')
      .agg(recordings=('subject', 'size'), hr_bpm=('ref_hr_bpm', 'median'),
           sdnn_ms=('ref_sdnn_ms', 'median'), rmssd_ms=('ref_rmssd_ms', 'median'),
           kept=('ref_kept_share', 'median'))
      .reindex(list(ACTIVITIES)).round(2).to_string())

70 recordings, 254352 raw frames, read in 20 s | frames that are not one 8-bit frame by size: 0 | frames dropped by the camera counter: 0
frame rate on the camera clock 30.0167-30.0180 fps | oximeter rate 59.9202-60.0336 Hz | samples lost 2803
the reference starts 0.05-0.24 s after the first frame; it ends before the last frame by more than 1 s in 3 recordings [{'subject': '003', 'activity': 'reading', 'ref_short_by_s': 8.4}, {'subject': '009', 'activity': 'stationary', 'ref_short_by_s': 3.2}, {'subject': '015', 'activity': 'reading', 'ref_short_by_s': 7.4}]
references that carry HRV: 58 of 70 (NB_P3_33 section 5: 58)
            recordings  hr_bpm  sdnn_ms  rmssd_ms  kept
activity                                               
stationary          12   79.13    42.43     39.53  0.97
reading             12   76.22    51.11     48.63  0.96
watching             8   74.34    53.97     41.40  0.99
talking             13   81.69    64.95     40.62  0.98
deep                13   74.89    78.2

## 2. Store schema and the single-recording walkthrough

### Schema (Rice iPPG)
| name | source | shape, type | notes |
|------|--------|-------------|-------|
| `frames` | raw frames, padded-box crop | [T, 72, 72, 3] uint8 | the training contract's geometry; the border is replicated where the box leaves the frame |
| `frames_clamped` | raw frames, box clamped | [T, 72, 72, 3] uint8 | what the app's own box takes today |
| `bvp` | oximeter waveform | [T] float32 | on the frame grid, with `offset_ms` applied: the label |
| `bvp_covered` | - | [T] bool | false where the oximeter had stopped, or had not started |
| `region_rgb` | raw frames, three regions | [T, 3, 3] float32 | forehead, left cheek, right cheek; mean R, G, B from the mosaic |
| `frame_t_ms` | camera clock | [T] float64 | ms after the first frame, on the PC clock |
| `ox_ppg`, `ox_t_ms`, `ox_missing` | oximeter | [N] each | the native 60 Hz record, its rebuilt clock, and which samples were interpolated |
| `ref_beat_ms`, `ref_beat_kept` | oximeter | [B], [B-1] | reference beat times, and which intervals the rules kept |

Key attributes: `role='heldout'`, `activity`, `has_hrv`; `fps` from the camera clock beside
`fps_nominal`; the crop box, `box_side_lost` (the store convention, worse axis) beside
`app_frac_side_lost` (what the app's framing gate reads) and the four padding amounts;
`offset_ms` with its anchor, median beat distance and matched share; the reference's own
numbers (`ref_hr_bpm`, `ref_sdnn_ms`, `ref_rmssd_ms`, `ref_kept_share`, `ref_short_by_s`); the
app's verdicts (`framing_verdict`, `face_checks`, `motion_verdict`, `motion_crop_px`); and the
quality readings `qc_pos_hr`, `qc_pos_hr_error` and `qc_pos_bvp_r`/`qc_pos_bvp_lag_s` for the
clamped crop, with `qc_pos_hr_padded` and `qc_pos_hr_error_padded` for the other. There is
**no `cardiac_sqi`** - see the guards above.

`usable` gates on exposure only, as in every Phase-3 store. A recording the framing gate
refuses is still written, flagged, because which crop to use is the open question this store
exists to answer.

### The label's polarity
`qc_pos_bvp_r` is the most negative correlation between the app's POS and the stored label
within 0.25 s, with its lag. POS correlates negatively with a stored `bvp` in every Phase-3
store, so the search is for that sign: the opposite-sign lobe half a beat away reaches the edge
of the window with nearly the same magnitude, and taking the larger magnitude has twice made a
sound label look flipped.

In [8]:
def demosaic(raw):
    return cv2.cvtColor(raw, BAYER_CODE)


def box_audit(row):
    """The app's held box and its own gates on one recording's detection samples."""
    samples = [demosaic(read_raw(frame_path(row.folder, i)))
               for i in detection_positions(int(row.frames))]
    box, found, faces = stable_box(samples, LANDMARKER)
    if box is None:
        raise RuntimeError('no face in any detection sample')
    framing, report, _ = framing_verdict(box, FRAME_W, FRAME_H)
    checks, _, summary = face_verdict(faces, box)
    subject = np.array([f[0] for f in faces if f])
    return box, {'found': int(found), 'of': len(samples),
                 'face_w': float(np.median(subject[:, 2] - subject[:, 0])),
                 'face_h': float(np.median(subject[:, 3] - subject[:, 1])),
                 'framing': framing, 'app_frac_side_lost': float(report['frac_side_lost']),
                 'aspect_in_training_range': bool(report['aspect_in_training_range']),
                 'face_checks': checks, 'second_face': int(summary['second_face_samples']),
                 'face_inside_min': float(summary['face_inside_min'])}


def box_side_lost(box):
    """The share of the box's side that falls outside the frame, on the worse axis - the store
    convention, each axis against its own side."""
    lost_x = max(0.0, -box[0]) + max(0.0, box[2] - 1.0)
    lost_y = max(0.0, -box[1]) + max(0.0, box[3] - 1.0)
    return float(max(lost_x / (box[2] - box[0]), lost_y / (box[3] - box[1])))


def padded_geometry(box, width=FRAME_W, height=FRAME_H):
    """The box in whole pixels, and the border each side needs to hold the part that leaves the
    frame."""
    px = (int(box[0] * width), int(box[1] * height), int(box[2] * width), int(box[3] * height))
    border = (max(0, -px[0]), max(0, -px[1]), max(0, px[2] - width), max(0, px[3] - height))
    return px, border


def crop_padded(rgb, px, border, size=TARGET_SIZE):
    """The whole box as a square crop, the frame's edge pixels replicated where the box leaves it.

    The face keeps the share of the crop it has in training, which clamping the box to the frame
    would change; what enters instead of the missing pixels is a static replicated border, with no
    pulse of its own.
    """
    x0, y0, x1, y1 = px
    left, top, right, bottom = border
    h, w = rgb.shape[:2]
    inner = rgb[max(0, y0):min(h, y1), max(0, x0):min(w, x1)]
    if left or top or right or bottom:
        inner = cv2.copyMakeBorder(inner, top, bottom, left, right, cv2.BORDER_REPLICATE)
    return cv2.resize(inner, (size, size), interpolation=cv2.INTER_AREA)


def region_rectangles(box, face_w, face_h):
    """Each region's rows and columns on the mosaic, on whole 2x2 cells, or None where fewer than
    MIN_REGION_CELLS cells fall inside the frame. The regions sit on the landmark box: centred
    where the held box is centred, with the median landmark width and height of the samples."""
    left = (box[0] + box[2]) / 2 - face_w / 2
    top = (box[1] + box[3]) / 2 - face_h / 2
    out = {}
    for name, (a, b, c, d) in REGIONS.items():
        c0, c1 = (int(np.clip((left + f * face_w) * FRAME_W, 0, FRAME_W)) // 2 * 2 for f in (a, c))
        r0, r1 = (int(np.clip((top + f * face_h) * FRAME_H, 0, FRAME_H)) // 2 * 2 for f in (b, d))
        cells = max(0, r1 - r0) * max(0, c1 - c0) // 4
        out[name] = (slice(r0, r1), slice(c0, c1)) if cells >= MIN_REGION_CELLS else None
    return out


def region_colours(raw, rectangles):
    """Mean R, G and B of each region straight from the mosaic, each colour over its own sites. A
    demosaiced frame interpolates each colour from the same sites, so over thousands of cells the
    means agree, without the interpolation's edges."""
    out = np.full((len(rectangles), 3), np.nan)
    for k, place in enumerate(rectangles.values()):
        if place is None:
            continue
        patch = raw[place]
        for c, sites in enumerate(SITES.values()):
            out[k, c] = np.mean([patch[s[0]::2, s[1]::2].mean() for s in sites])
    return out


def frame_pass(row, box, rectangles):
    """One read per frame, three uses: the regions' colours from the mosaic, and the two crops.

    Frames are read, demosaiced and cropped in parallel threads: file reads and OpenCV's calls
    release the interpreter's lock.
    """
    n = int(row.frames)
    px, border = padded_geometry(box)
    frames = np.empty((n, TARGET_SIZE, TARGET_SIZE, 3), dtype=np.uint8)
    clamped = np.empty_like(frames)
    regions = np.empty((n, len(rectangles), 3), dtype=np.float32)

    def one(i):
        raw = read_raw(frame_path(row.folder, i))
        rgb = demosaic(raw)
        return i, crop_padded(rgb, px, border), crop_resize(rgb, box), region_colours(raw, rectangles)

    with ThreadPoolExecutor(FRAME_WORKERS) as pool:
        for i, padded, clamp, colours in pool.map(one, range(n)):
            frames[i], clamped[i], regions[i] = padded, clamp, colours
    return frames, clamped, regions


def nearest_gap(cam, ref):
    """Signed distance from each reference beat to the nearest camera beat, camera minus
    reference."""
    j = np.clip(np.searchsorted(cam, ref), 1, cam.size - 1)
    left, right = cam[j - 1] - ref, cam[j] - ref
    return np.where(np.abs(left) <= np.abs(right), left, right)


def align_offset(cam_t, ref_t, borders, centre, half_width=ALIGN_HALF_WIDTH_S, step=0.0025):
    """The offset (finger minus camera) within half_width of the subject's anchor whose reference
    beats sit closest to a camera beat, with its median distance and matched share.

    Only reference beats that border an interval the rules kept are used, and only those every
    candidate offset keeps inside the camera's span.
    """
    offsets = np.arange(centre - half_width, centre + half_width + step / 2, step)
    if cam_t.size < 3:
        return np.nan, np.nan, np.nan
    ref = ref_t[borders & (ref_t >= cam_t[0] + offsets[-1] + MATCH_TOL_S)
                & (ref_t <= cam_t[-1] + offsets[0] - MATCH_TOL_S)]
    if ref.size < 3:
        return np.nan, np.nan, np.nan
    gaps = np.abs(np.array([nearest_gap(cam_t + o, ref) for o in offsets]))
    median = np.median(gaps, axis=1)
    i = int(np.argmin(median))
    return float(offsets[i]), float(median[i]), float(np.mean(gaps[i] <= MATCH_TOL_S))


def camera_beats_on_clock(frames_u8, fps, frame_t):
    """The app's POS pulse, inverted so the systolic rise is a rising edge, and the PC times of its
    steepest rises."""
    pulse = -pos(rgb_trace(frames_u8), fps)
    if not np.any(pulse):
        return pulse, np.array([])
    positions, _ = steepest_rises(pulse, fps, dominant_period(pulse, fps))
    return pulse, np.interp(positions, np.arange(frame_t.size), frame_t)


def signed_lag(video, reference, fps, max_lag_s=0.25):
    """(r, lag in s) at the lag within +/-max_lag_s where r is most negative.

    POS and a stored bvp have opposite polarity in every Phase-3 store, so the matching
    alignment is the most negative correlation. Taking the largest magnitude instead is
    ambiguous: the opposite-sign lobe half a beat away reaches the edge of the search with
    nearly the same magnitude, which is how it has twice been mistaken for a flipped label.
    """
    span = int(round(max_lag_s * fps))
    lags = np.arange(-span, span + 1)
    rs = np.array([np.corrcoef(video[max(0, -k):len(video) - max(0, k)],
                               reference[max(0, k):len(reference) - max(0, -k)])[0, 1]
                   for k in lags])
    if np.all(np.isnan(rs)):
        return float("nan"), float("nan")
    i = int(np.nanargmin(rs))
    return float(rs[i]), float(lags[i] / fps)


def plain(value):
    """A value HDF5 attributes accept: numpy scalars become Python ones, None becomes NaN."""
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return float(value)
    return float('nan') if value is None else str(value)


def reading_of(frames_u8, fps):
    """The app's own heart-rate reading from a crop, as a flat dictionary."""
    result = HRSpectral().estimate(frames_u8, fps)
    detail = result.detail or {}
    return {'hr': float(result.value), 'status': result.status,
            'confidence': float(result.confidence),
            'usable_fraction': float(detail.get('usable_fraction', 0.0)),
            'kept': int(sum(detail.get('window_kept', []))),
            'windows': int(detail.get('n_total', 0))}


def build_record(key):
    """Builds one unified-schema record for a Rice iPPG recording from its frames and oximeter."""
    t_start = time.perf_counter()
    row = recordings.loc[(recordings.subject == key[0]) & (recordings.activity == key[1])].iloc[0]
    ref = REF[key]
    frame_t, fps = ref['frame_t'], ref['fps']
    box, gates = box_audit(row)
    rectangles = region_rectangles(box, gates['face_w'], gates['face_h'])
    frames, clamped, regions = frame_pass(row, box, rectangles)
    if len(frames) != frame_t.size:
        raise RuntimeError(f'{len(frames)} frames against {frame_t.size} clock entries')

    # The app reads the clamped crop, so its reading and the alignment reproduce NB_P3_33 part 2.
    app_read = reading_of(clamped, fps)
    padded_read = reading_of(frames, fps)
    motion, motion_note, motion_summary = motion_verdict(clamped, fps)
    pulse, cam_beats = camera_beats_on_clock(clamped, fps, frame_t)
    borders = np.zeros(ref['beat_t'].size, dtype=bool)
    borders[1:] |= ref['beat_kept']
    borders[:-1] |= ref['beat_kept']
    anchor = ANCHOR_MS[key[0]] / 1000.0
    offset, gap_s, matched = align_offset(cam_beats, ref['beat_t'], borders, anchor)
    used = anchor if not np.isfinite(offset) else offset

    # The reference on the frame grid: the finger's waveform at each frame's time plus the offset,
    # which is where that frame's beat is recorded at the finger.
    bvp = np.interp(frame_t + used, ref['grid_t'], ref['grid']).astype(np.float32)
    covered = (frame_t + used >= ref['grid_t'][0]) & (frame_t + used <= ref['grid_t'][-1])
    readout_band = butter(4, [HR_MIN_BPM / 60.0, 3.5], btype='band', fs=fps, output='sos')
    # POS as the app computes it, not the inverted copy the beats are found on, so the sign of r
    # means the same here as in every other Phase-3 store.
    pos_r, pos_lag = signed_lag(sosfiltfilt(readout_band, -pulse),
                                sosfiltfilt(readout_band, bvp - bvp.mean()), fps)
    brightness = float(np.median([frames[i].mean()
                                  for i in np.linspace(0, len(frames) - 1, 9).astype(int)]))
    skin = regions.reshape(-1, 3)
    skin = np.nanmedian(skin[np.isfinite(skin).all(axis=1)], axis=0)

    arrays = {'frames': frames, 'frames_clamped': clamped, 'bvp': bvp,
              'bvp_covered': covered, 'region_rgb': regions,
              'frame_t_ms': ((frame_t - frame_t[0]) * 1000.0).astype(np.float64),
              'ox_ppg': ref['grid'].astype(np.float32),
              'ox_t_ms': ((ref['grid_t'] - frame_t[0]) * 1000.0).astype(np.float64),
              'ox_missing': ref['missing'],
              'ref_beat_ms': ((ref['beat_t'] - frame_t[0]) * 1000.0).astype(np.float64),
              'ref_beat_kept': ref['beat_kept']}
    attrs = dict(
        schema_version=SCHEMA_VERSION, dataset='Rice-iPPG', role='heldout',
        subject_id=key[0], subject_uid=f'rice_{key[0]}', activity=key[1],
        source_dir=row.folder.name,
        fps=float(fps), fps_nominal=30.0, n_frames=int(len(frames)), frame_size=int(TARGET_SIZE),
        duration_s=float(frame_t[-1] - frame_t[0]), camera_frames_dropped=int(ref['dropped']),
        crop_rule='padded frame, border replicated; frames_clamped holds the app box clamped',
        box_rx0=float(box[0]), box_ry0=float(box[1]), box_rx1=float(box[2]), box_ry1=float(box[3]),
        box_side_lost=box_side_lost(box),
        pad_left=int(padded_geometry(box)[1][0]), pad_top=int(padded_geometry(box)[1][1]),
        pad_right=int(padded_geometry(box)[1][2]), pad_bottom=int(padded_geometry(box)[1][3]),
        bayer_pattern=PATTERN, demosaic='cv2.COLOR_BayerBG2RGB',
        skin_r=float(skin[0]), skin_g=float(skin[1]), skin_b=float(skin[2]),
        skin_r_g=float(skin[0] / skin[1]), skin_b_g=float(skin[2] / skin[1]),
        regions=','.join(REGIONS), regions_read=int(sum(p is not None
                                                        for p in rectangles.values())),
        has_frames=True, has_frames_clamped=True, has_bvp=True, has_region_rgb=True,
        has_hrv=bool(key not in CLOCK_DOUBTED | TOO_THIN),
        has_ecg=False, has_resp=False, has_spo2=False, has_eda=False, has_hr=False,
        bvp_units_note='CMS50D+ finger waveform, 7 bits of 0-127, on its rebuilt 60 Hz clock, '
                       'interpolated to the frame times with offset_ms applied',
        ox_fs=float(OX_FS), ox_rate_hz=float(ref['row']['ox_rate_hz']),
        ox_lost=int(ref['row']['ox_lost']), ox_delay_med_ms=float(ref['row']['ox_delay_med_ms']),
        ref_starts_after_s=float(ref['row']['ref_starts_after_s']),
        ref_short_by_s=float(ref['row']['ref_short_by_s']),
        ref_covered_frames=int(covered.sum()),
        ref_beats=int(ref['beat_t'].size), ref_kept_share=float(ref['row']['ref_kept_share']),
        ref_kept_s=float(ref['row']['ref_kept_s']),
        ref_hr_bpm=float(ref['metrics']['hr_bpm']), ref_sdnn_ms=float(ref['metrics']['sdnn_ms']),
        ref_rmssd_ms=float(ref['metrics']['rmssd_ms']),
        offset_ms=float(used * 1000.0), offset_anchor_ms=float(ANCHOR_MS[key[0]]),
        offset_found=bool(np.isfinite(offset)), offset_gap_ms=float(gap_s * 1000.0),
        offset_matched=float(matched),
        detections=int(gates['found']), detections_attempted=int(gates['of']),
        face_w=float(gates['face_w']), face_h=float(gates['face_h']),
        face_checks=gates['face_checks'], face_second=int(gates['second_face']),
        face_inside_min=float(gates['face_inside_min']),
        framing_verdict=gates['framing'], app_frac_side_lost=float(gates['app_frac_side_lost']),
        aspect_in_training_range=bool(gates['aspect_in_training_range']),
        motion_verdict=motion, motion_note=motion_note or '',
        motion_crop_px=float((motion_summary or {}).get('amp_median', float('nan'))),
        motion_periodic=float((motion_summary or {}).get('clear_frac', float('nan'))),
        brightness=brightness, frame_ok=bool(FRAME_DARK <= brightness <= FRAME_BRIGHT),
        usable=bool(FRAME_DARK <= brightness <= FRAME_BRIGHT),
        qc_pos_hr=app_read['hr'], qc_pos_status=app_read['status'],
        qc_pos_confidence=app_read['confidence'], qc_pos_kept=app_read['kept'],
        qc_pos_windows=app_read['windows'],
        qc_pos_hr_error=float(app_read['hr'] - ref['metrics']['hr_bpm']),
        qc_pos_hr_padded=padded_read['hr'], qc_pos_status_padded=padded_read['status'],
        qc_pos_confidence_padded=padded_read['confidence'],
        qc_pos_hr_error_padded=float(padded_read['hr'] - ref['metrics']['hr_bpm']),
        qc_pos_bvp_r=pos_r, qc_pos_bvp_lag_s=pos_lag,
        seconds_to_build=round(time.perf_counter() - t_start, 1))
    return arrays, {k: plain(v) for k, v in attrs.items()}


DEMO = ('003', 'deep')
demo_arrays, demo_attrs = build_record(DEMO)
print('record:', '_'.join(DEMO))
print(json.dumps(demo_attrs, indent=2, default=str))
print({k: (tuple(v.shape), str(v.dtype)) for k, v in demo_arrays.items()})
print('the two crops differ, as the box leaves the frame:',
      not np.array_equal(demo_arrays['frames'], demo_arrays['frames_clamped']))

record: 003_deep
{
  "schema_version": "p3_v1",
  "dataset": "Rice-iPPG",
  "role": "heldout",
  "subject_id": "003",
  "subject_uid": "rice_003",
  "activity": "deep",
  "source_dir": "Subj_003_deep",
  "fps": 30.017876398194495,
  "fps_nominal": 30.0,
  "n_frames": 3829,
  "frame_size": 72,
  "duration_s": 127.52273464202881,
  "camera_frames_dropped": 0,
  "crop_rule": "padded frame, border replicated; frames_clamped holds the app box clamped",
  "box_rx0": -0.03270993530750277,
  "box_ry0": 0.15430977046489713,
  "box_rx1": 1.0026065796613692,
  "box_ry1": 1.1896262854337691,
  "box_side_lost": 0.18315779058104806,
  "pad_left": 41,
  "pad_top": 0,
  "pad_right": 3,
  "pad_bottom": 194,
  "bayer_pattern": "RGGB",
  "demosaic": "cv2.COLOR_BayerBG2RGB",
  "skin_r": 148.69921875,
  "skin_g": 135.71774291992188,
  "skin_b": 81.77692413330078,
  "skin_r_g": 1.095650553703308,
  "skin_b_g": 0.6025514602661133,
  "regions": "forehead,cheek_left,cheek_right",
  "regions_read": 3,
  "has_fr

## 3. Write function and batch processing
Each recording becomes a group named `<subject>_<activity>` in one consolidated HDF5, both
frame datasets in `(32, 72, 72, 3)` chunks and uncompressed, every other dataset unchunked.
Append mode with skip-existing makes the run resumable, and each group is flushed as soon as it
is written. A recording reads about 4.6 GB of raw frames from the T7 and takes 12-15 s, so the
full batch runs in about twenty minutes and the store comes to roughly 7.5 GiB, two crops
included. `MAX_RECORDS` limits a trial run.

The log goes to `Data/processing_log_rice_ippg_phase3.csv`, and carries recording-level numbers
only - no frames and no waveforms.

In [9]:
LOG_COLS = ('subject_id', 'activity', 'n_frames', 'fps', 'usable', 'frame_ok', 'detections',
            'box_side_lost', 'app_frac_side_lost', 'framing_verdict', 'face_checks',
            'motion_verdict', 'motion_crop_px', 'brightness', 'has_hrv', 'ref_hr_bpm',
            'ref_rmssd_ms',
            'ref_kept_share', 'ref_short_by_s', 'offset_ms', 'offset_gap_ms', 'offset_matched',
            'qc_pos_hr', 'qc_pos_status', 'qc_pos_hr_error', 'qc_pos_hr_error_padded',
            'qc_pos_bvp_r', 'qc_pos_bvp_lag_s', 'seconds_to_build', 'status', 'error')


def write_group(store, name, arrays, attrs):
    """Writes one recording as a group: both frame datasets chunked along time, then every
    attribute."""
    group = store.create_group(name)
    for key, value in arrays.items():
        if key.startswith('frames'):
            # Whole-frame chunks along time let a clip read touch a few chunks and decompress
            # nothing. The length is clamped because h5py rejects chunks longer than the data.
            chunks = (min(FRAMES_PER_CHUNK, len(value)),) + value.shape[1:]
            group.create_dataset(key, data=value, chunks=chunks)
        else:
            group.create_dataset(key, data=value)
    for key, value in attrs.items():
        group.attrs[key] = value


def process_all(keys, out_h5, limit=None, overwrite=False):
    """Writes every recording into one consolidated HDF5, one group each.

    Append mode with skip-existing makes the run resumable, and a flush after each group keeps
    finished groups on disk if a later one fails.
    """
    keys = list(keys)[:limit] if limit else list(keys)
    rows = []
    out_h5.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    with h5py.File(out_h5, 'a') as store:
        for i, key in enumerate(keys, 1):
            name = f'{key[0]}_{key[1]}'
            if name in store and not overwrite:
                attrs = dict(store[name].attrs)
                attrs.update(status='skipped', error='')
            else:
                try:
                    arrays, attrs = build_record(key)
                    if name in store:
                        del store[name]
                    write_group(store, name, arrays, attrs)
                    store.flush()
                    attrs = dict(attrs, status='ok', error='')
                except Exception as exc:
                    attrs = {'subject_id': key[0], 'activity': key[1], 'usable': False,
                             'status': 'error', 'error': str(exc)}
                print(f'   {i}/{len(keys)} {name} -> {attrs["status"]}'
                      f'{" (" + attrs["error"] + ")" if attrs["error"] else ""} | '
                      f'{(time.perf_counter() - t0) / 60:.1f} min', flush=True)
            rows.append({c: attrs.get(c) for c in LOG_COLS})
    return pd.DataFrame(rows)


ORDER = sorted(REF, key=lambda k: (k[0], ACTIVITIES.index(k[1])))
log = process_all(ORDER, OUT_H5, limit=MAX_RECORDS, overwrite=OVERWRITE)
log.to_csv(LOG_PATH, index=False)
print('store:', OUT_H5, f'| {OUT_H5.stat().st_size / 2**30:.2f} GiB')
print('ok:', int((log.status == 'ok').sum()), '| skipped:', int((log.status == 'skipped').sum()),
      '| errors:', int((log.status == 'error').sum()))
print('usable:', int(log.usable.fillna(False).astype(bool).sum()), '/', len(log))
print('log ->', LOG_PATH)

store: D:\QualityPhys\phase 3 benchmarks\rice_ippg_phase3_rc_none.h5 | 7.42 GiB
ok: 0 | skipped: 70 | errors: 0
usable: 70 / 70
log -> D:\code\QualityPhys - CRVSE Project\Data\processing_log_rice_ippg_phase3.csv


## 4. Output validation
Re-opens the store and checks every group against the schema, then the things that make it
trustworthy rather than merely complete:

- **The guards hold.** The training loader's own filter is applied to the store; it must admit
  nothing.
- **NB_P3_33 is reproduced.** The face found in all 840 detection samples, the framing gate
  refusing all 70 recordings, the app's POS reading within 3.6 bpm of the finger in 69 of them,
  and 58 references carrying HRV. A difference means the code or the input moved, and nothing
  downstream should run until it is explained.
- **The label sits where it should**: POS against `bvp` negative, at a lag near zero now that
  each recording's offset is applied.
- **What the padded crop changes**, recorded here because that is the comparison the store was
  built to allow.

These are checks on the stored data, not an evaluation of any method.

In [10]:
EXPECTED_SETS = {'frames', 'frames_clamped', 'bvp', 'bvp_covered', 'region_rgb', 'frame_t_ms',
                 'ox_ppg', 'ox_t_ms', 'ox_missing', 'ref_beat_ms', 'ref_beat_kept'}
# NB_P3_33's findings, which this store has to reproduce: part 2 sections 10, 14 and 16.
NB_P3_33_POS_READ, NB_P3_33_POS_WORST_BPM = 69, 3.6
NB_P3_33_FRAMING_REFUSED, NB_P3_33_DETECTIONS = 70, 840


def training_filter(group):
    """NB_P3_07's own test: a group it would index must have frames, a bvp and an SQI above 0.05."""
    attrs = group.attrs
    return ('frames' in group and 'bvp' in group
            and float(attrs.get('cardiac_sqi', 0.0)) >= 0.05
            and bool(attrs.get('usable', False))
            and group['frames'].shape[0] >= CLIP_LEN)


with h5py.File(OUT_H5, 'r') as store:
    groups = sorted(store.keys())
    problems = []
    for name in groups:
        group = store[name]
        attrs = group.attrs
        if set(group.keys()) != EXPECTED_SETS:
            problems.append((name, 'datasets', sorted(set(group.keys()) ^ EXPECTED_SETS)))
        n = int(attrs['n_frames'])
        shapes = {k: group[k].shape for k in EXPECTED_SETS}
        if (shapes['frames'] != (n, TARGET_SIZE, TARGET_SIZE, 3)
                or shapes['frames_clamped'] != shapes['frames']
                or shapes['bvp'] != (n,) or shapes['bvp_covered'] != (n,)
                or shapes['region_rgb'] != (n, len(REGIONS), 3)
                or shapes['frame_t_ms'] != (n,)):
            problems.append((name, 'shapes', shapes))
        if group['frames'].chunks != (FRAMES_PER_CHUNK, TARGET_SIZE, TARGET_SIZE, 3):
            problems.append((name, 'chunks', group['frames'].chunks))
        if 'cardiac_sqi' in attrs or attrs['role'] != 'heldout':
            problems.append((name, 'guard', 'cardiac_sqi present or role not heldout'))
        if group['ref_beat_kept'].shape[0] != group['ref_beat_ms'].shape[0] - 1:
            problems.append((name, 'beats', group['ref_beat_ms'].shape))
    admitted = [name for name in groups if training_filter(store[name])]
    qc = pd.DataFrame([{k: store[name].attrs[k] for k in
                        ('subject_id', 'activity', 'fps', 'usable', 'has_hrv', 'detections',
                         'framing_verdict', 'face_checks', 'motion_verdict', 'motion_crop_px',
                         'box_side_lost', 'brightness', 'ref_hr_bpm', 'ref_rmssd_ms',
                         'offset_ms', 'offset_gap_ms', 'offset_matched', 'qc_pos_hr',
                         'qc_pos_status', 'qc_pos_hr_error', 'qc_pos_confidence',
                         'qc_pos_hr_padded', 'qc_pos_hr_error_padded',
                         'qc_pos_confidence_padded', 'qc_pos_bvp_r', 'qc_pos_bvp_lag_s')}
                       for name in groups])
    sample = groups[:: max(1, len(groups) // 5)]
    for name in sample:
        group = store[name]
        bvp, covered = group['bvp'][:], group['bvp_covered'][:]
        if not np.all(np.isfinite(bvp)) or covered.sum() == 0:
            problems.append((name, 'bvp', 'not finite, or no frame covered'))

print(f'groups: {len(groups)} | schema problems: {problems or "none"}')
print(f'the training loader admits {len(admitted)} of {len(groups)} groups '
      f'(it must admit none): {admitted or "none"}')
print(f'usable: {int(qc.usable.sum())}/{len(qc)} | carry HRV: {int(qc.has_hrv.sum())} | '
      f'face found in all {N_DETECT_FRAMES} samples: {int((qc.detections == N_DETECT_FRAMES).sum())}'
      f' ({int(qc.detections.sum())} of {N_DETECT_FRAMES * len(qc)} samples, NB_P3_33: '
      f'{NB_P3_33_DETECTIONS})')
print(f'the app\'s gates on the clamped crop: framing '
      f'{qc.framing_verdict.value_counts().to_dict()} (NB_P3_33 refused '
      f'{NB_P3_33_FRAMING_REFUSED}), face checks {qc.face_checks.value_counts().to_dict()}, '
      f'motion {qc.motion_verdict.value_counts().to_dict()}')
print(f'the box loses {qc.box_side_lost.min():.0%}-{qc.box_side_lost.max():.0%} of its side outside '
      f'the frame | brightness {qc.brightness.min():.0f}-{qc.brightness.max():.0f}')
read = qc[np.isfinite(qc.qc_pos_hr)]
print(f'\nthe app\'s POS reading against the reference: read in {len(read)}/{len(qc)} recordings '
      f'(NB_P3_33: {NB_P3_33_POS_READ}/70), worst error {read.qc_pos_hr_error.abs().max():.1f} bpm '
      f'(NB_P3_33: {NB_P3_33_POS_WORST_BPM}) | status {qc.qc_pos_status.value_counts().to_dict()}')
print(f'the same reading from the padded crop: read in '
      f'{int(np.isfinite(qc.qc_pos_hr_padded).sum())}, worst error '
      f'{qc.qc_pos_hr_error_padded.abs().max():.1f} bpm, confidence '
      f'{qc.qc_pos_confidence_padded.median():.3f} against {qc.qc_pos_confidence.median():.3f} '
      f'on the clamped crop')
print(f'POS against the stored label: r {qc.qc_pos_bvp_r.min():+.2f} to '
      f'{qc.qc_pos_bvp_r.max():+.2f} (negative is the Phase-3 convention), at a lag of '
      f'{qc.qc_pos_bvp_lag_s.min():+.3f} to {qc.qc_pos_bvp_lag_s.max():+.3f} s')
print(f'\nalignment: offsets {qc.offset_ms.min():+.0f} to {qc.offset_ms.max():+.0f} ms, '
      f'median distance to a camera beat {qc.offset_gap_ms.median():.1f} ms, matched '
      f'{qc.offset_matched.median():.3f} in the median recording')
print()
print(qc.groupby('activity')
      .agg(recordings=('subject_id', 'size'), hrv=('has_hrv', 'sum'),
           pos_error=('qc_pos_hr_error', lambda v: v.abs().median()),
           pos_error_padded=('qc_pos_hr_error_padded', lambda v: v.abs().median()),
           offset_ms=('offset_ms', 'median'), gap_ms=('offset_gap_ms', 'median'),
           motion_px=('motion_crop_px', 'median'))
      .reindex([a for a in ACTIVITIES if a in set(qc.activity)]).round(2).to_string())

groups: 70 | schema problems: none
the training loader admits 0 of 70 groups (it must admit none): none
usable: 70/70 | carry HRV: 58 | face found in all 12 samples: 70 (840 of 840 samples, NB_P3_33: 840)
the app's gates on the clamped crop: framing {'REJECT': 70} (NB_P3_33 refused 70), face checks {'ACCEPT': 70}, motion {'ACCEPT': 70}
the box loses 6%-34% of its side outside the frame | brightness 42-100

the app's POS reading against the reference: read in 69/70 recordings (NB_P3_33: 69/70), worst error 3.6 bpm (NB_P3_33: 3.6) | status {'ok': 67, 'degraded_capture': 2, 'insufficient_quality': 1}
the same reading from the padded crop: read in 65, worst error 4.3 bpm, confidence 0.845 against 0.848 on the clamped crop
POS against the stored label: r -0.93 to -0.38 (negative is the Phase-3 convention), at a lag of -0.100 to -0.033 s

alignment: offsets -82 to +30 ms, median distance to a camera beat 11.9 ms, matched 0.993 in the median recording

            recordings  hrv  pos_error  

## 5. Findings (2026-10-25)

**The store.** 70 of 70 recordings written, no errors: `phase 3 benchmarks\rice_ippg_phase3_rc_none.h5`, 7.42 GiB, 32 min of build time at a median 26 s per recording (21.7-40.8 s), both crops and the three region traces included. Every group passes the schema check: the two frame datasets at 3,387-4,019 frames of 72x72 in (32, 72, 72, 3) chunks, the frame-aligned arrays at the same length, the native oximeter record beside them, and one more beat time than kept intervals. All three regions were read in all 70 recordings with no missing value, so the region fusion of NB_P3_33 part 3 can be re-measured from the store alone. The batch cell's own line reads 70 skipped because the groups were written by an earlier pass of it; every figure here comes from the store itself.

**The guards hold.** The training loader's own filter admits 0 of 70: no group carries `cardiac_sqi`, every group carries `role='heldout'`, and the store sits in `phase 3 benchmarks\`, outside every training glob.

**Reproduction.** The face was found in all 840 detection samples, the framing gate refuses all 70 recordings, the app's POS reading is within 3.6 bpm of the finger in 69 of 70, and 58 references carry HRV - NB_P3_33's figures, reproduced by the code that wrote the store. The alignment reproduces to three decimals: the median distance from a reference beat to its nearest camera beat is 9.91, 10.33, 13.03, 22.32 and 11.62 ms on the five activities, against section 13's 9.914, 10.327, 13.032, 22.319 and 11.623. Every recording's offset was found inside its subject's window and none at its edge, at most 52.5 ms from the anchor, over a span of -82.5 to +30.0 ms.

**The two crops, measured for the first time.** The clamped crop - what the app takes today - reads 69 of 70 recordings, with a median absolute error of 0.44 bpm and a worst of 3.57. The crop the training contract asks for, with the frame padded out, reads 65, at 0.54 and 4.27. Where both read, the difference is slight and the same at every activity (0.26 against 0.28 bpm at rest, 1.15 against 1.08 in deep breathing), and their window confidence is indistinguishable (0.845 against 0.848). **The five recordings the padded crop loses are all talking ones** - 003, 004, 009, 010 and 013 - each refused for too few usable windows rather than low confidence. One recording, 009/talking, is refused on both crops. A plausible reason, untested here: the padded crop holds a wider view at the same 72 pixels, so the central fraction the colour trace averages reaches further into the mouth and jaw, which move while talking. Either way the clamped crop is not the weaker one on this dataset, and the choice between them stays open.

**The label's polarity, and a lag that is not an alignment error.** POS correlates negatively with the stored label in all 70 recordings, between -0.93 and -0.38, which is the convention of every other Phase-3 store. The lag of that correlation is -0.033 s in 32 recordings, -0.067 in 30 and -0.100 in 8 - the label leading the camera by one to three frames - although the beat fiducials of the two devices agree to a median 12 ms. The two measure different things: the beat offset aligns the steepest rises, while the correlation weighs the whole waveform, and a finger pulse and a POS trace do not have the same shape. It matters for a waveform loss or a waveform-level score, not for a rate. The weakest correlations are talking recordings (009 at -0.38, 010 and 012 at -0.61 and -0.62).

**Coverage.** The reference covers 99.8-99.9% of the frames in 67 recordings, the shortfall being the 0.05-0.24 s before the oximeter started. In the three recordings whose oximeter link failed early it covers 92.9% (003/reading), 93.9% (015/reading) and 97.1% (009/stationary) - the three NB_P3_33 named. `bvp_covered` marks those frames, and nothing else needs trimming.

**Colour and exposure.** The skin sits at R/G 0.98-1.11 (median 1.06) and B/G 0.58-0.72 (median 0.65) over the three regions, with green between 77 and 184 of 255; crop brightness runs 42-100 and the exposure gate passes everywhere. The darkest crop, 010/talking at 42, is also one the padded crop refuses.

**Ready for evaluation: yes, with one question deliberately left open.** Nothing about the reference, the clocks or the beats is unresolved, and both crops are in the store, so the crop rule can be settled by reading it rather than by another pass over 254,352 raw frames.

**Decision.**
- **Heart rate on all 70 recordings, HRV on the 58**, each scored over the frames `bvp_covered` marks, at the recording's own `fps`. Every figure states which crop produced it; nothing is pooled across crops.
- **Talking is reported separately**, not averaged in. It carries the weakest correlations, the widest beat distances (80.7 ms at worst), the only recording refused on both crops, and all five the padded crop refuses.
- **The crop rule follows from the evaluation, not from this notebook.** The clamped crop reads more often here; the padded one is the only one whose face fill and aspect match what the models trained on. An evaluation that reports both settles it on the figure that matters rather than on geometry alone.
- **The jitter correction of NB_P3_33 part 3 is re-measurable from the store**: the region traces are there, and the beats and offsets with them.
- **Next: the evaluation script**, reporting both crops, and the two corrections to `check_hrv_from_video.py` that NB_P3_33 named - alignment by the median distance between beats, and a minimum beat spacing tied to each recording's own beat period.